# Research Project Repository Testing

This notebook tests the `ResearchProjectRepository` and its complex Many-to-Many relationships with `Staff` and `Sample`.

In [ ]:
# Setup database and import required modules
from domain.database import Base, engine
from domain.config import DB_URL
from domain.repositories.unit_of_work import UnitOfWorkFactory

from domain.models import ResearchProject, Staff, Sample, Patient, SampleType, Container
from domain.repositories.research_project_repository import ResearchProjectRepository
from domain.repositories.staff_repository import StaffRepository
from domain.repositories.patient_repository import PatientRepository
from domain.repositories.sample_type_repository import SampleTypeRepository
from domain.repositories.base_repository import BaseRepository
from domain.repositories.sample_repository import SampleRepository

from datetime import date
import warnings
warnings.filterwarnings('ignore')


uow_factory = UnitOfWorkFactory(DB_URL)

# Pre-requisite data setup for relationships
with uow_factory.create() as uow:
    staff = Staff(code="S-500", name="Eleanor", lastname="Rigby", role="researcher", active=True)
    patient = Patient(code="P-500", name="Father", lastname="McKenzie", birth_date=date(1950, 1, 1), active=True, test="Blood Test")
    stype = SampleType(type_name="Serum")
    container = Container(code="C-500", type_name="Flask")
    
    StaffRepository(uow.session).save(staff)
    PatientRepository(uow.session).save(patient)
    SampleTypeRepository(uow.session).save(stype)
    BaseRepository[Container, int](uow.session, Container).save(container)
    uow.commit()
    
    sample = Sample(code="SMP-500", volume=2.5, extraction_date=date(2024, 3, 1), status="pending", 
                    id_patient=patient.id, id_sample_type=stype.id, id_container=container.id)
    SampleRepository(uow.session).save(sample)
    uow.commit()
    
    print("Pre-requisite data (Staff and Sample) created successfully.")

## 1. Research Project Repository Tests

In [ ]:
# Test CRUD and specific queries for ResearchProjectRepository
with uow_factory.create() as uow:
    project_repo = ResearchProjectRepository(uow.session)
    staff_repo = StaffRepository(uow.session)
    sample_repo = SampleRepository(uow.session)

    # --- CREATE ---
    print("--- CREATE RESEARCH PROJECTS ---")
    rp1 = ResearchProject(project_name="Oncology Alpha", start_date=date(2024, 1, 1), description="Phase 1")
    rp2 = ResearchProject(project_name="Genetics Beta", start_date=date(2024, 6, 15), description="Phase 2")
    
    project_repo.save(rp1)
    project_repo.save(rp2)
    uow.commit()
    print(f"Created: {rp1.project_name}")
    print(f"Created: {rp2.project_name}\n")

    # --- ADD STAFF AND SAMPLES (RELATIONSHIPS) ---
    print("--- TEST RELATIONSHIPS ---")
    # Using the specific domain operation from StaffRepository to add staff to a project
    staff_repo.add_to_research_project(code="S-500", project_name="Oncology Alpha", role="principal_investigator")
    uow.commit()
    print("Staff S-500 added to 'Oncology Alpha' via ProjectTeam.")
    
    # Appending sample to the project relationship directly via the model
    sample = sample_repo.get_by_code("SMP-500")
    rp1.samples.append(sample)
    uow.commit()
    print("Sample SMP-500 linked to 'Oncology Alpha' via ResearchProjectSamples.\n")

    # --- SPECIFIC QUERIES ---
    print("--- SPECIFIC QUERIES ---")
    found_project = project_repo.get_by_name("Oncology Alpha")
    print(f"get_by_name('Oncology Alpha'): ID {found_project.id}")
    
    # Test Eager Loading method
    full_project = project_repo.get_with_team(rp1.id)
    print(f"get_with_team({rp1.id}):")
    print(f"  -> Loaded {len(full_project.staff_members)} staff member(s)")
    print(f"  -> Loaded {len(full_project.samples)} sample(s)\n")
    
    # Test inverse eager load from StaffRepository
    staff_roles = staff_repo.get_research_project_with_roles("S-500")
    print(f"Staff S-500 project roles: {len(staff_roles)} assignment(s) found.")

    # --- UPDATE ---
    print("\n--- UPDATE ---")
    rp1.description = "Phase 1 - Active"
    uow.commit()
    print(f"Updated description to: {project_repo.get_by_id(rp1.id).description}\n")

    # --- DELETE ---
    print("--- DELETE ---")
    project_repo.delete(rp2)
    uow.commit()
    print(f"Deleted 'Genetics Beta'. Total projects remaining: {project_repo.count()}")